# KubeCon India 2026 — Environment Setup (AWS / Crossplane track)

The notebook form of [`init-with-xp.sh`](init-with-xp.sh) — Track 1 of the
**"One Interface To Rule Them All"** demo: a local cluster + KubeVela + **Crossplane**
as the cloud-resource orchestrator for AWS S3.

Unlike a monolithic notebook, each step here **reuses the repo's reusable helpers**
in [`../../../scripts/`](../../../scripts/) (the same functions `init-with-xp.sh`
calls) — so the notebook and the script never drift.

## Prerequisites
- `k3d`, `kubectl`, `helm`, `vela`, `docker`, `python3`
- AWS credentials in `aws-setup/.env.aws` (created as a template on first run if
  missing — fill it in, then re-run that step)

## Steps
0. Check prerequisites (tools + `config.yaml`)
1. Set up the Python venv + load configuration (writes `.env.sh`)
2. Create the k3d cluster + local registry
3. Establish AWS connectivity (`.env.aws`)
4. Install Crossplane + wire the AWS S3 provider
5. Install the KubeVela control plane (+ VelaUX)

**Run the cells in order.** Each `%%bash` cell is self-contained (it re-sources the
helpers), so you can also re-run an individual step after fixing an issue.

> After this notebook, run [`setup-with-xp.sh`](setup-with-xp.sh) (or its notebook)
> to apply the `bucket` backing + deploy the sample app.


## Step 0: Prerequisites

Check the required CLIs are on `PATH` and this folder has its `config.yaml`. Also
make sure PyYAML is available to the kernel (used to preview the config).

In [ ]:
# Prerequisites check
import shutil, os, sys, subprocess

print("=== Checking prerequisites ===\n")

tools = ["k3d", "kubectl", "helm", "vela", "docker", "python3"]
missing = [t for t in tools if shutil.which(t) is None]
for t in tools:
    print(f"  {'OK ' if shutil.which(t) else 'MISSING'}  {t}")

print("\nconfig.yaml:", "OK" if os.path.exists("config.yaml") else "MISSING")

try:
    import yaml  # noqa: F401
    print("PyYAML:     OK")
except ImportError:
    print("PyYAML:     installing into the kernel...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyyaml"])
    print("PyYAML:     OK")

if missing:
    print("\n*** Install the missing tools before continuing:")
    print("    brew install k3d kubectl helm docker python@3.12")
    print("    curl -fsSl https://kubevela.io/script/install.sh | bash   # vela")
elif not os.path.exists("config.yaml"):
    print("\n*** config.yaml is missing from this folder.")
else:
    print("\nAll prerequisites satisfied.")


: 

## Step 1: Python venv + load configuration

Create/activate the demo-local venv (installs PyYAML for `load_config.py`), then run
the repo's `load_config` — it parses this folder's `config.yaml`, exports
`CLUSTER_NAME` / `API_PORT` / `HTTP_PORT` / `CROSSPLANE_NAMESPACE` / `MIN_CRDS`, and
writes `.env.sh` next to it (which every later cell sources).

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"   # notebook runs from aws-setup/
source "$REPO_ROOT/scripts/common.sh"
source "$REPO_ROOT/scripts/setup-venv.sh"
source "$REPO_ROOT/scripts/load-config.sh"

# Demo-local venv (installs PyYAML) — activated for THIS cell so load_config.py finds it.
setup_venv "$PWD/.venv" "$REPO_ROOT/scripts/requirements.txt"

# Parse config.yaml → export vars + write .env.sh next to it (sourced by later cells).
load_config "$PWD/config.yaml"

echo
echo "Config: CLUSTER_NAME=$CLUSTER_NAME  API_PORT=$API_PORT  HTTP_PORT=$HTTP_PORT"
echo "        CROSSPLANE_NAMESPACE=$CROSSPLANE_NAMESPACE  MIN_CRDS=$MIN_CRDS"


## Step 2: Create the k3d cluster + local registry

`create_cluster` deletes any existing cluster/registry, frees the registry port,
creates the registry then the cluster (wired together), switches the kubectl context,
and verifies access. Uses the `CLUSTER_NAME` / `API_PORT` / `HTTP_PORT` from Step 1.

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
source "$REPO_ROOT/scripts/common.sh"
source .env.sh
source "$REPO_ROOT/scripts/create-cluster.sh"

create_cluster


## Step 3: Establish AWS connectivity

`load_aws_env` sources `aws-setup/.env.aws` and exports `AWS_*`. If the file is
missing it writes a template and **stops** — fill in your real credentials, then
re-run this cell. (These are the platform credentials Crossplane uses to provision
S3.)

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
source "$REPO_ROOT/scripts/common.sh"
source "$REPO_ROOT/scripts/load-aws-env.sh"

# No --skip: stops (writes a template) if .env.aws is missing so creds get filled in.
load_aws_env "$PWD/.env.aws"


## Step 4: Install Crossplane + wire the AWS S3 provider

Mirrors `init-with-xp.sh` Phase 3, in dependency order:
1. install Crossplane and wait for its CRDs,
2. create the `aws-credentials` secret (from `.env.aws`),
3. apply the `function-patch-and-transform` function,
4. apply the providers (aws-s3 + kubernetes),
5. apply the ProviderConfigs.

This cell re-loads `.env.aws` first, since each `%%bash` cell is a fresh shell.

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
source "$REPO_ROOT/scripts/common.sh"
source .env.sh
source "$REPO_ROOT/scripts/load-aws-env.sh"
source "$REPO_ROOT/scripts/install-crossplane.sh"
source "$REPO_ROOT/scripts/create-aws-secret.sh"
source "$REPO_ROOT/scripts/apply-crossplane-function.sh"
source "$REPO_ROOT/scripts/apply-crossplane-provider.sh"
source "$REPO_ROOT/scripts/apply-crossplane-provider-config.sh"

load_aws_env "$PWD/.env.aws"          # re-export AWS_* into this cell

install_crossplane
wait_for_crossplane_crds
create_aws_secret "$CROSSPLANE_NAMESPACE" aws-credentials
apply_crossplane_function       "$REPO_ROOT/platform/crossplane/function"
apply_crossplane_provider       "$REPO_ROOT/platform/crossplane/provider"
apply_crossplane_provider_config "$REPO_ROOT/platform/crossplane/provider-config"
# The S3 XRD + Composition are applied later by setup-with-xp.sh Phase 1.


## Step 5: Install the KubeVela control plane

`install_kubevela --velaux` installs `vela-core`, waits for it, enables the VelaUX
addon, and backgrounds a port-forward to http://localhost:8000. Drop `--velaux` for
the control plane only.

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
source "$REPO_ROOT/scripts/common.sh"
source .env.sh
source "$REPO_ROOT/scripts/install-kubevela.sh"

install_kubevela --velaux


## Setup complete

The AWS / Crossplane track is bootstrapped: k3d cluster + local registry, Crossplane
+ the AWS S3 provider, and the KubeVela control plane (+ VelaUX at
http://localhost:8000).

### Next steps
- Apply the `bucket` backing + deploy the app: run [`setup-with-xp.sh`](setup-with-xp.sh).
- Inspect: `kubectl get pods -A`, `kubectl get providers,functions -A`.
- Swap backends (same claim): `vela def apply ../../../platform/kubevela/components/bucket-ack.cue` (ACK),
  or bootstrap the GCP track in [`../gcp-setup/`](../gcp-setup/) (KCC).

### Teardown
- ACK track only: `./teardown-with-ack.sh` (empties the S3 buckets first).
- Then delete the cluster: `../cleanup.sh`.
